# Elbow method 
This method helps user pick the right number of colors/clusters for image compression. It's one of two most popular methods, the other one is *Silhouette Method*.

**In this notebook you can find visualizations and explaination and testing, for implementation go to `elbow.py`**.

In [ ]:
import sys

sys.path.append("..")
import matplotlib.pyplot as plt

import elbow
from image_io import image_load

Firstly, let's start with loading the image and reshaping it's size to:

$$
\begin{matrix}
(r_1, g_1, b_1), \\
(r_2, g_2, b_2), \\
(r_3, g_3, b_3), \\
\cdots, \\
(r_n, g_n, b_n)
\end{matrix}
$$
where $n$ is the number of all pixels ($\text{height of an image} \cdot \text{width of an image}$).

In [ ]:
pixels, _ = image_load("../examples/small-landscape.jpg")
X = pixels.reshape(-1, 3)

## How this method works?
### Step 1.
Apply k-means-clustering algorithm and form the clusters for different values of $k$.
For example: $k =[ 1,2,3,4,5,6,7,8,\ldots ,17 ]$
### Step 2.
For each of those values calculate the **"within clusters sum of squares" (WCSS)**.
$$ \text{WCSS} =  \sum^{K}_{k=1} \sum_{d_i \in C_k} ||d_i - \mu_k||^2$$
where:
- $K$ is the number of clusters
- $C_k$ is the set of points in cluster $k$
- $d_i$ is data point in each cluster
- $\mu_k$ is the centroid of cluster $k$

> WCSS is a metric to measure how compact the clusters are, the lower WCSS, the more compact clusters are.
> It represents the total squared distance between each data point and the centroid of clusters it belongs to. 

In [ ]:
k_values = list(range(1, 17))
inertias = elbow.compute_elbow_curve(X, k_values, seed=1)

In [ ]:
print(inertias)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_values, inertias, marker="o")
plt.plot(k_values[3], inertias[3], marker="D", color="green", markersize=14)
plt.xlabel("K")
plt.ylabel("Inertia (WCSS)")
plt.title("Elbow method")
plt.xticks(k_values)
plt.grid(alpha=0.3)
plt.show()

### Step 3.
Now we have to find the **elbow**. It's clearly visible that on plot above the elbow is in $k=4$ (green diamond), but we need to calculate it with formula.

>The elbow is the point that is the furthest from 0 first and last k value.

1. Normalize both axes to $[0,1]$
$$ \hat{\text{val}}_i = \frac{\text{val}_i - \min(\text{val})}{\max(\text{val}) - \min(\text{val})} $$

2. Define the line from the first point to last
$$ P_1 = (\hat{k}_1, \hat{\text{wcss}}_1) \quad P_2 = (\hat{k}_n, \hat{\text{wcss}}_n) $$

3. Calculate perpendicular distance from each point to that line

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_values, inertias, marker="o")
plt.plot([k_values[0], k_values[-1]], [inertias[0], inertias[-1]])
plt.plot(4, inertias[3], marker="D", color="green", markersize=12)
plt.plot([k_values[3], k_values[-1]], [inertias[3], inertias[-1]], color="red")
plt.plot([k_values[0], k_values[3]], [inertias[0], inertias[3]], color="purple")
plt.xlabel("K")
plt.ylabel("Inertia (WCSS)")
plt.title("Elbow method")
plt.xticks(k_values)
plt.grid(alpha=0.3)

To do so we need to calculate the area of this triangle (*orange, purple, red*).
$$ \text{Area} = \frac{1}{2} \cdot \text{base} \cdot \text{height} \implies \text{height} = 2\cdot\frac{\text{Area}}{\text{base}}$$
- $\text{base} = \sqrt{(x_1 - x_2)^2 + (y_1- y_2)^2}$ is a distance between $P_1, P_2$
- $\text{Area} = \sqrt{s(s-a)(s-b)(s-c)}$, where $s=\frac{1}{2}(a+b+c)$ is semiperemeter and $a,b,c$ are distances between each points (*Heron's formula*)

### Step 4. 
Pick the K with the largest distance.
$$ K^{*} = \argmax(\text{height}_i)_{k_i} $$

In [ ]:
optimal_k = elbow.find_elbow_k(k_values, inertias)
print(f"Optimal k value is: {optimal_k}")